# 01 - Hello World: 基本設定與 LLM 呼叫

本筆記本示範如何在此框架中進行最基本的操作。

## 本筆記本涵蓋內容

本範例將逐步示範以下兩個核心操作：

1. **載入設定（Config Loading）**：使用 `init_config()` 從 `config/config.yaml` 載入專案設定，並檢視設定內容。
2. **初始化並呼叫 LLM Client**：使用 `LLMClient` 連接 LLM 服務，發送對話請求並解析回應結果。

> **前置條件**：請確保已安裝相依套件（`uv sync`），並已在 `config/config.yaml` 中填入正確的 API 設定。

In [ ]:
# 標準函式庫
import sys
import os

# 將專案根目錄加入 Python 路徑
sys.path.insert(0, os.path.abspath(".."))

# 專案模組
from app.utils.config import init_config, get_config
from llm_service import LLMClient, LLMResponse

print("匯入完成。")

## 載入設定

使用 `init_config()` 初始化全域設定單例（singleton）。此函式會讀取 `config/config.yaml`，並支援 `${ENV_VAR:default}` 語法解析環境變數。

初始化後即可透過 `get_config()` 在任何地方取得相同的設定物件，無需重複傳遞。

In [ ]:
# 初始化設定（預設讀取 config/config.yaml）
cfg = init_config()

# 檢視設定內容
print("=== 目前設定內容 ===")
for key, value in cfg.items():
    print(f"  {key}: {value}")

## 初始化 LLM Client

使用 `LLMClient.load_config()` 從設定檔建立客戶端實例。`LLMClient` 內部會自動讀取所需的連線與模型設定。

In [ ]:
# 建立 LLM Client 實例
client = LLMClient.load_config()

print("LLMClient 初始化完畢。")

## 呼叫 LLM

使用 `client.chat()` 發送對話請求。此方法接受 `system_prompt` 與 `user_prompt` 兩個參數，並回傳 `LLMResponse` 物件。

`LLMResponse` 包含以下欄位：
- `content`：模型的文字回應
- `model`：實際使用的模型名稱
- `usage`：token 使用量統計
- `latency_ms`：請求延遲時間（毫秒）

> **注意**：此步驟需要一個正在運行的 LLM 服務（例如本地的 Ollama 或遠端 API）。若服務未啟動，將會捕捉例外並顯示錯誤訊息，不會中斷程式執行。

In [ ]:
system_prompt = "你是一個有用的助手。"
user_prompt = "請用一句話介紹自己。"

try:
    response: LLMResponse = client.chat(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
    )

    print("=== LLM 回應 ===")
    print(f"內容     : {response.content}")
    print(f"模型     : {response.model}")
    print(f"Token 用量: {response.usage}")
    print(f"延遲     : {response.latency_ms:.1f} ms")

except Exception as e:
    print(f"[警告] LLM 呼叫失敗：{e}")
    print("請確認 LLM 服務已啟動，並在 config/config.yaml 中填入正確的 API 設定。")

## 小結

本筆記本示範了此框架的兩個基本操作：

| 步驟 | 函式 / 類別 | 說明 |
|------|------------|------|
| 載入設定 | `init_config()` / `get_config()` | 載入 YAML 設定，支援環境變數解析 |
| 呼叫 LLM | `LLMClient.load_config()` / `client.chat()` | 建立客戶端並發送對話請求 |

### 後續步驟

- 查看 `02_custom_workflow.ipynb` 了解如何使用 LangGraph 建構多步驟工作流程。
- 查看 `03_evaluation.ipynb` 了解如何進行模型評估。
- 查看 `04_mlflow_tracking.ipynb` 了解如何整合 MLflow 進行實驗追蹤。
- 修改 `config/config.yaml` 以連接您自己的 LLM 服務端點。